**Code for deriving anatomical & functional "meanings" from learned HMM model states.**

Ultimately consists of different "levels" of explanation:

- First / top level (mandatory): "Global", i.e. what feature patterns define each model state globally, across subjects?
- Second / lower level (pseudo-optional): "Subject-wise", i.e. how do different individuals express model states? e.g. some subjects may exhibit the abstract model states more "purely" or more "strongly" than others.

The second level is usually where clinically-relevant insights arise. The "global" level is defining the "common language" the brain uses, whereas the subject-level analyses helps determine how different individuals "speak" and "use" the language differently.



**Currently outputs the following tables:**
- 'subject_session_state_fidelity'
- 'subject_session_state_profiles_long'
- 'state_report_long'
- 'state_report_top_features'

The first two tables above contain our primary ML inputs; the latter two are global metrics only (i.e. pertain to decoding/interpreting the underlying HMM model's feature space), and as such are diagnostic/interpretive in nature.

Or, put another way, the first two are subject-level features that act as clinical/trait predictors; whereas the latter two define the state space of the model and facilitate their characterization in anatomical/functional/scientific terms.

**Final output** is saved into the target directory as '_flattened_HMM_state_features_ALL.csv'

[Runtime: Trivial]

----------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, subprocess
import time
from pathlib import Path
import json
import pandas as pd
import numpy as np
import joblib
import re
from numpy.linalg import norm

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARAMETERS:

# General parameters:
HARD_STOP    = config['hard_errors']
RANDOM_SEED  = config['random_seed']


### PROCESSING PARAMETERS:

OVERWRITE = config['overwrite_state_definitions']

EXPORT_LEVEL = config['HMM_decoding']['export_level'].strip().lower()
if EXPORT_LEVEL not in {"session", "subject"}:
    raise ValueError(f"[INIT ERROR] HMM_decoding.export_level must be 'session' or 'subject' (got: {EXPORT_LEVEL})")


# __________________________________________________________________________________________________________
### SET FILEPATHS:

BASE_DIRECTORY = Path(config['root_output_directory'])

RUN_MANIFEST_PATH    = BASE_DIRECTORY / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH = BASE_DIRECTORY / 'fMRI_manifest.csv'


### INPUTS:

# Grab dataset-selection config variables:
DATASET_SELECTOR    = str(config["ML_training"]["dataset_selector"]).strip().lower()
DATASET_MANUAL_PATH = config["ML_training"].get("dataset_path", None)

# Original "dataset pointer" is in this directory, and it also contains the raw data ('X_train', 'X_all', etc.):
DATA_DIR = Path(BASE_DIRECTORY) / config['ML_prep']['training_data_dir']

# Actual models live here (under a directory w/ the same name as governs the pointer logic):
MODELS_DIR = Path(BASE_DIRECTORY) / config['HMM_training']['HMM_model_directory']


### OUTPUTS:

ROOT_OUTPUT_DIR = Path(BASE_DIRECTORY) / config['interpretation_output_dir']


# __________________________________________________________________________________________________________
### INITIALIZATION:

RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs    = pd.read_csv(fMRI_PARAMETERS_PATH)

-----------

Locate & load core data inputs:

In [ ]:
# __________________________________________________________________________________________________________
### RESOLVE DATASET + LOAD FINAL_MODEL BUNDLE + LOCATE X_all
#
# Creates:
#   - DATASET_NAME, DATASET_DIR
#   - DATASET_MODELS_DIR
#   - FINAL_DIR
#   - final_model, final_pca (or None)
#   - feature_columns_expected (or None)
#   - x_all_path, provenance_path
#   - final_selection_payload (dict, best-effort)
# __________________________________________________________________________________________________________

def handle_error(message: str):
    if HARD_STOP:
        raise RuntimeError(message)
    else:
        print(f"[WARN] {message}")

# Resolve 'DATASET_NAME' and 'DATASET_DIR':
DATASET_NAME = None
DATASET_DIR = None

if DATASET_SELECTOR == "latest":
    pointer_path = Path(DATA_DIR) / "LATEST_DATASET.json"
    if not pointer_path.exists():
        raise FileNotFoundError(
            f"[INIT ERROR] dataset_selector='latest' but pointer file not found:\n  {pointer_path}")
    with open(pointer_path, "r") as f:
        pointer = json.load(f)
    pointer_dataset_dir = Path(pointer.get("dataset_dir", "")).expanduser()
    if pointer_dataset_dir is None or str(pointer_dataset_dir).strip() == "":
        raise RuntimeError(
            f"[INIT ERROR] Pointer file exists but missing/empty 'dataset_dir':\n  {pointer_path}")
    DATASET_NAME = pointer_dataset_dir.name
    DATASET_DIR = Path(DATA_DIR) / DATASET_NAME  # <-- enforces rooting under 'DATA_DIR'
elif DATASET_SELECTOR == "manual":
    if DATASET_MANUAL_PATH is None or str(DATASET_MANUAL_PATH).strip() == "":
        raise ValueError(
            "[INIT ERROR] dataset_selector='manual' but ML_training.dataset_path is empty.")
    manual_dir = Path(str(DATASET_MANUAL_PATH)).expanduser()
    DATASET_NAME = manual_dir.name
    DATASET_DIR = Path(DATA_DIR) / DATASET_NAME  # <-- enforces rooting under 'DATA_DIR'
else:
    raise ValueError(
        f"[INIT ERROR] ML_training.dataset_selector must be 'latest' or 'manual' "
        f"(got: {DATASET_SELECTOR})")
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"[INIT ERROR] DATASET_DIR not found: {DATASET_DIR}")

# Resolve dataset-specific models directory & 'FINAL_MODEL' directory
DATASET_MODELS_DIR = Path(MODELS_DIR) / DATASET_NAME
if not DATASET_MODELS_DIR.exists():
    raise FileNotFoundError(
        "[INIT ERROR] Dataset models directory not found.\n"
        f"  Expected: {DATASET_MODELS_DIR}")

FINAL_DIR = DATASET_MODELS_DIR / "FINAL_MODEL"
if not FINAL_DIR.exists():
    raise FileNotFoundError(
        "[INIT ERROR] FINAL_MODEL directory not found.\n"
        f"  Expected: {FINAL_DIR}\n"
        "Run the cross-model evaluation script and export a final-final model first.")

# Expected FINAL_MODEL artifacts:
final_model_path = FINAL_DIR / "final_model.joblib"
final_pca_path = FINAL_DIR / "final_pca.joblib"
feature_cols_path = FINAL_DIR / "feature_columns.json"
final_sel_path = FINAL_DIR / "final_model_selection.json"

if not final_model_path.exists():
    raise FileNotFoundError(f"[INIT ERROR] Missing required file: {final_model_path}")

# Load model (+ optional PCA) & metadata:
try:
    final_model = joblib.load(final_model_path)
except Exception as exc:
    raise RuntimeError(f"[INIT ERROR] Failed to load final_model.joblib: {exc}")

final_pca = None
if final_pca_path.exists():
    try:
        final_pca = joblib.load(final_pca_path)
    except Exception as exc:
        handle_error(f"[INIT WARN] Failed to load final_pca.joblib; proceeding without PCA. Error: {exc}")
        final_pca = None

feature_columns_expected = None
if feature_cols_path.exists():
    try:
        with open(feature_cols_path, "r") as f:
            tmp = json.load(f)
        if isinstance(tmp, dict) and isinstance(tmp.get("feature_columns", None), list) and tmp["feature_columns"]:
            feature_columns_expected = list(tmp["feature_columns"])
    except Exception as exc:
        handle_error(f"[INIT WARN] Failed to read feature_columns.json; will infer from X_all/provenance. Error: {exc}")

final_selection_payload = None
if final_sel_path.exists():
    try:
        with open(final_sel_path, "r") as f:
            final_selection_payload = json.load(f)
    except Exception as exc:
        handle_error(f"[INIT WARN] Failed to read final_model_selection.json. Error: {exc}")

# Resolve frozen data inputs ('X_all' + provenance sidecar):
x_all_path = DATASET_DIR / "X_all.csv"
provenance_path = DATASET_DIR / "provenance.json"

missing = []
if not x_all_path.exists():
    missing.append(str(x_all_path))
if not provenance_path.exists():
    # Note: provenance is useful, but not strictly required if 'feature_columns.json' exists:
    handle_error(f"[INIT WARN] provenance.json not found (will proceed): {provenance_path}")

if missing:
    raise FileNotFoundError("[INIT ERROR] Missing required frozen-data file(s):\n  - " + "\n  - ".join(missing))

# Print reports:
print("\n[INIT] Dataset + FINAL_MODEL bundle resolved:")
print(f"   DATASET_SELECTOR     = '{DATASET_SELECTOR}'")
print(f"   DATASET_NAME         = {DATASET_NAME}\n")
print(f"   DATASET_DIR          = {DATASET_DIR}")
print(f"     X_all.csv          = {x_all_path}")
print(f"     provenance.json    = {provenance_path} {'(missing)' if not provenance_path.exists() else ''}\n")
print(f"   DATASET_MODELS_DIR   = {DATASET_MODELS_DIR}")
print(f"   FINAL_DIR            = {FINAL_DIR}")
print(f"     final_model.joblib = {final_model_path}")
print(f"     final_pca.joblib   = {final_pca_path} {'(not present)' if not final_pca_path.exists() else ''}")
print(f"     feature_columns    = {feature_cols_path} {'(not present)' if not feature_cols_path.exists() else ''}")
print(f"     selection.json     = {final_sel_path} {'(not present)' if not final_sel_path.exists() else ''}")
if final_selection_payload and isinstance(final_selection_payload, dict):
    print("\n[INIT] Final selection summary:")
    print(f"   selected_protocol = {final_selection_payload.get('selected_protocol', None)}")
    print(f"   selected_K        = {final_selection_payload.get('selected_K', None)}")
    print(f"   PCA enabled       = {final_selection_payload.get('protocol_config_effective', {}).get('PCA', {}).get('enabled', None)}")

In [ ]:
# __________________________________________________________________________________________________________
### LOAD X_all + PREPARE FEATURE MATRICES + DECODE POSTERIOR STATE PROBABILITIES (FROM SCRATCH):

# Load 'X_all.csv':
x_all_df = pd.read_csv(x_all_path)

required_identifier_columns = ["subject_ID", "session_ID", "time_index"]
missing_identifier_columns = [c for c in required_identifier_columns if c not in x_all_df.columns]
if missing_identifier_columns:
    raise RuntimeError(
        "[DATA ERROR] X_all.csv missing required identifier columns: "
        + ", ".join(missing_identifier_columns))

# Resolve feature columns & validate schema:
if not feature_columns_expected or not isinstance(feature_columns_expected, list):
    raise RuntimeError(
        "[DATA ERROR] feature_columns_expected is missing/empty. "
        "Script #13 requires feature_columns.json to enforce correct feature ordering.")

missing_feature_columns = [c for c in feature_columns_expected if c not in x_all_df.columns]
if missing_feature_columns:
    raise RuntimeError(
        "[DATA ERROR] X_all.csv is missing feature columns required by the final model. "
        f"Missing {len(missing_feature_columns)} columns (showing up to 20):\n"
        + "\n".join(missing_feature_columns[:20]))

# Enforce exact feature ordering, as expected by the model bundle:
identifier_df = x_all_df.loc[:, required_identifier_columns].copy()

feature_matrix_raw = x_all_df.loc[:, feature_columns_expected].to_numpy(dtype=float)

print(f"\n[DATA] Loaded X_all.csv: shape = {x_all_df.shape}")
print(f"[DATA] Raw feature matrix shape (pre-PCA) = {feature_matrix_raw.shape} | dtype = {feature_matrix_raw.dtype}")

# Apply PCA (if present):
if final_pca is not None:
    try:
        feature_matrix_model = final_pca.transform(feature_matrix_raw)
    except Exception as exc:
        raise RuntimeError(f"[PCA ERROR] Failed to apply final_pca.transform() to feature_matrix_raw: {exc}")
    print(f"[PCA]  Applied PCA transform: {feature_matrix_raw.shape} -> {feature_matrix_model.shape}")
else:
    feature_matrix_model = feature_matrix_raw
    print("[PCA] No PCA found. Using raw feature matrix as model input.")

# Decode posterior probabilities & "hard" state labels:
has_predict_proba = hasattr(final_model, "predict_proba")
has_predict = hasattr(final_model, "predict")

if not has_predict_proba and not has_predict:
    raise RuntimeError(
        "[MODEL ERROR] final_model has neither predict_proba() nor predict(). "
        "Cannot decode states.")

if has_predict_proba:

    # ----------------------------------------------------------------------------------
    # IMPORTANT: Respect sequence boundaries for posterior decoding (don't use all data as contiguous sequence)!
    #    Build:
    #      (a) a deterministic row ordering by [subject_ID, session_ID, time_index], then
    #      (b) compute lengths per sequence, and (c) decode using hmmlearn lengths=...
    #    Finally, map decoded posteriors back to original row order so downstream code is unchanged.
    # ----------------------------------------------------------------------------------

    # Add explicit row index so we can map sorted outputs back to original order:
    identifier_with_row_index_df = identifier_df.copy()
    identifier_with_row_index_df["_row_index"] = np.arange(identifier_with_row_index_df.shape[0], dtype=int)

    # Deterministic sort order for defining contiguous sequences:
    sorted_id_df = identifier_with_row_index_df.sort_values(
        ["subject_ID", "session_ID", "time_index"],
        ascending=[True, True, True],
        kind="mergesort")  # <-- stable sort

    sorted_row_indices = sorted_id_df["_row_index"].to_numpy(dtype=int)

    # Compute sequence lengths in this sorted order:
    # (each (subject_ID, session_ID) group is one independent HMM sequence)
    lengths_series = (
        sorted_id_df
        .groupby(["subject_ID", "session_ID"], dropna=False)
        .size())
    lengths = lengths_series.to_list()

    # Sanity checks:
    if int(np.sum(lengths)) != int(feature_matrix_model.shape[0]):
        raise RuntimeError(
            "[MODEL ERROR] Sequence lengths do not sum to total rows.\n"
            f"  sum(lengths) = {int(np.sum(lengths))}\n"
            f"  feature_matrix_model rows = {int(feature_matrix_model.shape[0])}")

    # Decode in sorted (sequence-contiguous) order using lengths:
    feature_matrix_model_sorted = feature_matrix_model[sorted_row_indices, :]

    try:
        posterior_sorted = final_model.predict_proba(feature_matrix_model_sorted, lengths=lengths)
    except TypeError:
        # If the model's 'predict_proba()' function doesn't accept lengths (unlikely for hmmlearn), we fall back and warn loudly:
        posterior_sorted = final_model.predict_proba(feature_matrix_model_sorted)
        print("[WARN] final_model.predict_proba() did not accept lengths=...; posteriors may cross sequence boundaries.")
    except Exception as exc:
        raise RuntimeError(f"[MODEL ERROR] predict_proba() failed during sequence-aware decoding: {exc}")

    hard_sorted = np.argmax(posterior_sorted, axis=1).astype(int)

    # Map back to original row order:
    posterior_probability_matrix = np.zeros_like(posterior_sorted, dtype=float)
    hard_state_label_array = np.zeros_like(hard_sorted, dtype=int)

    posterior_probability_matrix[sorted_row_indices, :] = posterior_sorted
    hard_state_label_array[sorted_row_indices] = hard_sorted

    print(
        f"\n[MODEL] Sequence-aware decoding used lengths for "
        f"{len(lengths)} sequences | mean length={float(np.mean(lengths)):.2f} | "
        f"min={int(np.min(lengths))} | max={int(np.max(lengths))}")

else:
    posterior_probability_matrix = None
    hard_state_label_array = final_model.predict(feature_matrix_model).astype(int)

# Infer K setting:
if posterior_probability_matrix is not None:
    number_of_states = int(posterior_probability_matrix.shape[1])
else:
    # hmmlearn usually has 'n_components'; fallback to 'max label + 1':
    number_of_states = int(getattr(final_model, "n_components", int(hard_state_label_array.max()) + 1))

print("\n[MODEL] Decoding complete:")
print(f"  predict_proba() available       = {has_predict_proba}")
print(f"  predict() available             = {has_predict}")
print(f"  number_of_states (K)            = {number_of_states}")
print(f"  hard_state_label_array shape    = {hard_state_label_array.shape}")
if posterior_probability_matrix is not None:
    print(f"  posterior_probability_matrix    = {posterior_probability_matrix.shape}   <--(rows x K)")

----------

Next, analyses of global state definitions:

In [ ]:
# __________________________________________________________________________________________________________
### GLOBAL STATE INTERPRETATION: POSTERIOR-WEIGHTED FEATURE PROFILES IN RAW FEATURE SPACE
#
# Creates:
#   - global_feature_mean_series
#   - global_feature_std_series
#   - state_feature_mean_df                (wide: states x features)
#   - state_feature_zscore_df              (wide: states x features; z relative to global)
#   - state_feature_profile_long_df        (long: one row per state x feature, with mean/diff/zscore)
#
# Notes:
#   - Uses posterior_probability_matrix (soft assignments) when available.
#   - Falls back to hard_state_label_array if posteriors are missing.
#   - Operates in RAW feature space (feature_matrix_raw), regardless of PCA usage for decoding.
# __________________________________________________________________________________________________________

# Input sanity-checks:
if feature_columns_expected is None or not isinstance(feature_columns_expected, list) or len(feature_columns_expected) == 0:
    raise RuntimeError("[INTERPRET ERROR] feature_columns_expected is missing/empty.")

if feature_matrix_raw is None or not isinstance(feature_matrix_raw, np.ndarray):
    raise RuntimeError("[INTERPRET ERROR] feature_matrix_raw is missing or not a numpy array.")

if feature_matrix_raw.shape[1] != len(feature_columns_expected):
    raise RuntimeError(
        "[INTERPRET ERROR] feature_matrix_raw column count does not match feature_columns_expected.\n"
        f"  feature_matrix_raw.shape[1] = {feature_matrix_raw.shape[1]}\n"
        f"  len(feature_columns_expected) = {len(feature_columns_expected)}")

number_of_windows = int(feature_matrix_raw.shape[0])

if posterior_probability_matrix is not None:
    if posterior_probability_matrix.shape[0] != number_of_windows:
        raise RuntimeError(
            "[INTERPRET ERROR] posterior_probability_matrix row count does not match feature_matrix_raw.\n"
            f"  posterior_probability_matrix.shape[0] = {posterior_probability_matrix.shape[0]}\n"
            f"  feature_matrix_raw.shape[0] = {number_of_windows}")
    if posterior_probability_matrix.shape[1] != number_of_states:
        raise RuntimeError(
            "[INTERPRET ERROR] posterior_probability_matrix number_of_states mismatch.\n"
            f"  posterior_probability_matrix.shape[1] = {posterior_probability_matrix.shape[1]}\n"
            f"  number_of_states = {number_of_states}")

# Compute global (all-window) feature means and standard deviations:
feature_df = pd.DataFrame(feature_matrix_raw, columns=feature_columns_expected)

global_feature_mean_series = feature_df.mean(axis=0)
# Use ddof=0 to define "population" std across all windows (stable for z-scoring)
global_feature_std_series = feature_df.std(axis=0, ddof=0).replace(0.0, np.nan)

# Compute posterior-weighted state means, in RAW feature space:
if posterior_probability_matrix is not None:
    responsibility_matrix = posterior_probability_matrix.astype(float)  # shape: (T, K)
else:
    # Hard-assignment fallback == build a 0/1 responsibility matrix:
    responsibility_matrix = np.zeros((number_of_windows, number_of_states), dtype=float)
    hard_state_labels = hard_state_label_array.astype(int)
    for state_index in range(number_of_states):
        responsibility_matrix[:, state_index] = (hard_state_labels == state_index).astype(float)
state_responsibility_sums = responsibility_matrix.sum(axis=0)  # length K

# Important guardrail!: avoid divide-by-zero if a state is never used (should be rare, but possible):
epsilon = 1e-12
state_responsibility_sums_safe = np.where(state_responsibility_sums > 0, state_responsibility_sums, np.nan)

# Compute weighted means per state:
#     --> i.e. mean_k = sum_t (gamma_tk * x_t) / sum_t gamma_tk
state_feature_mean_matrix = (responsibility_matrix.T @ feature_matrix_raw) / (state_responsibility_sums_safe[:, None] + epsilon)
state_feature_mean_df = pd.DataFrame(
    state_feature_mean_matrix,
    index=[f"state_{state_index:02d}" for state_index in range(number_of_states)],
    columns=feature_columns_expected)

### Compute differences & z-score-normalized state profiles (relative to global state definitions):
# difference from global mean:
state_feature_difference_df = state_feature_mean_df.subtract(global_feature_mean_series, axis=1)
# z-score relative to global distribution across all windows:
state_feature_zscore_df = state_feature_difference_df.divide(global_feature_std_series, axis=1)

# Summarize state usage (i.e. "global responsibility mass"):
state_occupancy_soft = state_responsibility_sums / np.nansum(state_responsibility_sums)
state_occupancy_df = pd.DataFrame({
    "state_label": [f"state_{state_index:02d}" for state_index in range(number_of_states)],
    "responsibility_sum": state_responsibility_sums,
    "occupancy_soft": state_occupancy_soft})

# Build long-format table (for downstream ranking / reporting):
state_feature_profile_long_df = (
    state_feature_mean_df
    .reset_index()
    .rename(columns={"index": "state_label"})
    .melt(id_vars=["state_label"], var_name="feature_name", value_name="state_mean"))
state_feature_profile_long_df["global_mean"] = state_feature_profile_long_df["feature_name"].map(global_feature_mean_series.to_dict())
state_feature_profile_long_df["global_std"] = state_feature_profile_long_df["feature_name"].map(global_feature_std_series.to_dict())
state_feature_profile_long_df["difference_from_global_mean"] = (
    state_feature_profile_long_df["state_mean"] - state_feature_profile_long_df["global_mean"])
state_feature_profile_long_df["zscore_from_global_mean"] = (
    state_feature_profile_long_df["difference_from_global_mean"] / state_feature_profile_long_df["global_std"])

# Print reports:
print("\n[INTERPRET] Global state feature profiles computed (RAW feature space):")
print(f"  number_of_windows                  = {number_of_windows}")
print(f"  number_of_features                 = {len(feature_columns_expected)}")
print(f"  number_of_states (K)               = {number_of_states}")
print(f"  used_predict_proba                 = {posterior_probability_matrix is not None}")

print("\n[INTERPRET] Soft occupancy by state (sum responsibilities / total):")
print(state_occupancy_df[["state_label", "occupancy_soft"]].to_string(index=False))

# Print a quick preview of top |z| features per state (diagnostic only):
preview_top_n = 8
print(f"\n[INTERPRET] Top {preview_top_n} features by |zscore| per state (diagnostic preview):")
for state_label in state_feature_zscore_df.index:
    top_features = (
        state_feature_zscore_df.loc[state_label]
        .dropna()
        .abs()
        .sort_values(ascending=False)
        .head(preview_top_n)
        .index
        .tolist())
    print(f"  {state_label}: {', '.join(top_features)}")

In [ ]:
# __________________________________________________________________________________________________________
### BUILD + EXPORT "STATE REPORT" TABLES (GLOBAL)
#
# Requires (from prior cells):
#   - DATASET_NAME
#   - feature_columns_expected
#   - state_feature_profile_long_df         (state_label, feature_name, state_mean, global_mean, global_std,
#                                           difference_from_global_mean, zscore_from_global_mean)
#   - state_occupancy_df                    (state_label, responsibility_sum, occupancy_soft)
#
# Creates:
#   - OUTPUT_DIR
#   - state_report_long_df                  (long, one row per state x feature; includes parsed ROI/metric + occupancy)
#   - state_report_top_features_df          (compact, top +/- features per state)
#
# Saves:
#   - OUTPUT_DIR / "state_report_long.csv"
#   - OUTPUT_DIR / "state_report_top_features.csv"
# __________________________________________________________________________________________________________

# Resolve 'OUTPUT_DIR' (dataset-specific):
OUTPUT_DIR = Path(ROOT_OUTPUT_DIR) / str(DATASET_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

state_report_long_path = OUTPUT_DIR / "state_report_long.csv"
state_report_top_features_path = OUTPUT_DIR / "state_report_top_features.csv"

def write_csv_with_overwrite_guard(df: pd.DataFrame, csv_path: Path, overwrite: bool):
    if csv_path.exists() and not overwrite:
        print(f"[EXPORT] File exists; not overwriting (OVERWRITE=False): {csv_path}")
        return
    df.to_csv(csv_path, index=False)
    print(f"[EXPORT] Wrote: {csv_path} | shape={df.shape}")

# Build long-format state report (w/ parsed feature metadata):
state_report_long_df = state_feature_profile_long_df.copy()

# Merge occupancy information ("soft" occupancy is the main field of interest here):
occupancy_lookup_df = state_occupancy_df.loc[:, ["state_label", "occupancy_soft"]].copy()
state_report_long_df = state_report_long_df.merge(
    occupancy_lookup_df, on="state_label", how="left", validate="many_to_one")

# Parse feature names such as:(e.g. "ROI019_closenessCentrality" or "GLOBAL_Q"):
roi_pattern = re.compile(r"^ROI(?P<roi_index>\d{3})_(?P<metric_name>.+)$")

def parse_feature_name(feature_name: str) -> dict:
    if not isinstance(feature_name, str) or feature_name.strip() == "":
        return {"is_global_feature": True, "roi_index": np.nan, "roi_label": None, "metric_name": None, "feature_family": None}

    match = roi_pattern.match(feature_name)
    if match:
        roi_index = int(match.group("roi_index"))
        metric_name = match.group("metric_name")
        metric_name_lower = metric_name.lower()

        # Set "family" buckets to organize features:
        if "closeness" in metric_name_lower:
            feature_family = "closeness_centrality"
        elif "betweenness" in metric_name_lower:
            feature_family = "betweenness_centrality"
        elif "eigenvector" in metric_name_lower:
            feature_family = "eigenvector_centrality"
        elif "degree" in metric_name_lower:
            feature_family = "degree_centrality"
        else:
            feature_family = "other_roi_metric"
        return {
            "is_global_feature": False,
            "roi_index": roi_index,
            "roi_label": f"ROI{roi_index:03d}",
            "metric_name": metric_name,
            "feature_family": feature_family}

    # Not ROI-based; treat as global feature:
    return {
        "is_global_feature": True,
        "roi_index": np.nan,
        "roi_label": None,
        "metric_name": feature_name,
        "feature_family": "global_metric"}

parsed_feature_metadata_df = pd.DataFrame(
    [parse_feature_name(feature_name) for feature_name in state_report_long_df["feature_name"].tolist()])

state_report_long_df = pd.concat([state_report_long_df.reset_index(drop=True), parsed_feature_metadata_df], axis=1)

# Reorder columns for readability:
preferred_column_order = [
    "state_label",
    "occupancy_soft",
    "feature_name",
    "is_global_feature",
    "roi_label",
    "roi_index",
    "metric_name",
    "feature_family",
    "state_mean",
    "global_mean",
    "global_std",
    "difference_from_global_mean",
    "zscore_from_global_mean"]
existing_columns_in_order = [c for c in preferred_column_order if c in state_report_long_df.columns]
remaining_columns = [c for c in state_report_long_df.columns if c not in existing_columns_in_order]
state_report_long_df = state_report_long_df.loc[:, existing_columns_in_order + remaining_columns]

# Build compact "top features" report per state (top positive + top negative by z-score):
TOP_FEATURES_PER_DIRECTION = 10      # <-- can adjust later if desired; this is for interpretability, not model-fitting

top_rows = []
for state_label, state_df in state_report_long_df.groupby("state_label", dropna=False):

    # Exclude any features with missing z-scores:
    usable_df = state_df.dropna(subset=["zscore_from_global_mean"]).copy()

    top_positive_df = usable_df.sort_values("zscore_from_global_mean", ascending=False).head(TOP_FEATURES_PER_DIRECTION)
    top_negative_df = usable_df.sort_values("zscore_from_global_mean", ascending=True).head(TOP_FEATURES_PER_DIRECTION)

    for direction_label, subset_df in [("positive", top_positive_df), ("negative", top_negative_df)]:
        for rank_index, row in enumerate(subset_df.itertuples(index=False), start=1):
            top_rows.append({
                "state_label": getattr(row, "state_label"),
                "occupancy_soft": getattr(row, "occupancy_soft"),
                "direction": direction_label,
                "rank_within_direction": rank_index,
                # "feature_name": getattr(row, "feature_name"),
                "roi_label": getattr(row, "roi_label"),
                "metric_name": getattr(row, "metric_name"),
                # "feature_family": getattr(row, "feature_family"),
                "state_mean": getattr(row, "state_mean"),
                "global_mean": getattr(row, "global_mean"),
                "difference_from_global_mean": getattr(row, "difference_from_global_mean"),
                "zscore_from_global_mean": getattr(row, "zscore_from_global_mean")})

state_report_top_features_df = pd.DataFrame(top_rows)

# Sort by rank for each [state x direction]:
state_report_top_features_df = state_report_top_features_df.sort_values(
    ["state_label", "direction", "rank_within_direction"],
    ascending=[True, True, True]).reset_index(drop=True)

# --------------------------------------
# Export:
# --------------------------------------
print(f"\n[EXPORT] OUTPUT_DIR resolved to: {OUTPUT_DIR}")
write_csv_with_overwrite_guard(state_report_long_df, state_report_long_path, overwrite=OVERWRITE)
write_csv_with_overwrite_guard(state_report_top_features_df, state_report_top_features_path, overwrite=OVERWRITE)

# --------------------------------------
# Print report:
# --------------------------------------
print("\n[STATE REPORT] Long-format table:")
print(f"  shape = {state_report_long_df.shape}")
print("\n[STATE REPORT] Top-features table:")
print(f"  shape = {state_report_top_features_df.shape}")

# PREVIEW: top 3 ranked effects for each [state x direction]:
print("\nPREVIEW: Top 3 observed effects for each [state_label] x [direction]:")
state_report_top_features_df[state_report_top_features_df['rank_within_direction'] < 4]

-------------

Next up, across-subject / across-session state-expression analyses:

In [ ]:
# __________________________________________________________________________________________________________
### SUBJECT-/SESSION-LEVEL STATE EXPRESSION PROFILES (POSTERIOR-WEIGHTED) + FIDELITY TO GLOBAL STATE SIGNATURES
#
# Goal:
#   For each (subject_ID, session_ID) and each state, compute a posterior-weighted mean feature vector
#   in RAW feature space, then quantify how closely that subject/session's state profile matches the
#   GLOBAL state signature.
#
# Creates:
#   - subject_session_state_profile_long_df
#   - subject_session_state_fidelity_df
#
# Saves (guarded by OVERWRITE):
#   - OUTPUT_DIR / "subject_session_state_profiles_long.csv"
#   - OUTPUT_DIR / "subject_session_state_fidelity.csv"
# __________________________________________________________________________________________________________

# Preconditions & setup:
if posterior_probability_matrix is None:
    raise RuntimeError(
        "[SUBJECT-LEVEL ERROR] posterior_probability_matrix is missing. "
        "Subject-level state profiles require soft assignments (predict_proba).")
required_identifier_columns = ["subject_ID", "session_ID", "time_index"]
missing_identifier_columns = [column_name for column_name in required_identifier_columns if column_name not in identifier_df.columns]
if missing_identifier_columns:
    raise RuntimeError(
        "[SUBJECT-LEVEL ERROR] identifier_df missing required columns: "
        + ", ".join(missing_identifier_columns))
if "state_feature_mean_df" not in globals():
    raise RuntimeError("[SUBJECT-LEVEL ERROR] state_feature_mean_df not found. Run the global profile cell first.")
if "global_feature_std_series" not in globals():
    raise RuntimeError("[SUBJECT-LEVEL ERROR] global_feature_std_series not found. Run the global profile cell first.")

state_index_to_label = {state_index: f"state_{state_index:02d}" for state_index in range(number_of_states)}
global_feature_std_safe_series = global_feature_std_series.replace(0.0, np.nan)

identifier_with_row_index_df = identifier_df.copy()
identifier_with_row_index_df["_row_index"] = np.arange(identifier_with_row_index_df.shape[0], dtype=int)

grouped_sequences = (
    identifier_with_row_index_df
    .sort_values(["subject_ID", "session_ID", "time_index"])
    .groupby(["subject_ID", "session_ID"], dropna=False))

epsilon = 1e-12

# Helper similarity functions:
def safe_pearson_correlation(x: np.ndarray, y: np.ndarray) -> float:
    """Pearson correlation with NaN guardrails; returns NaN if insufficient variance."""
    if x.size == 0 or y.size == 0:
        return np.nan
    x = x.astype(float)
    y = y.astype(float)
    x_centered = x - np.nanmean(x)
    y_centered = y - np.nanmean(y)
    x_std = np.nanstd(x_centered, ddof=0)
    y_std = np.nanstd(y_centered, ddof=0)
    if not np.isfinite(x_std) or not np.isfinite(y_std) or x_std == 0.0 or y_std == 0.0:
        return np.nan
    return float(np.nanmean((x_centered / x_std) * (y_centered / y_std)))

def safe_cosine_similarity(x: np.ndarray, y: np.ndarray) -> float:
    """Cosine similarity with NaN guardrails."""
    if x.size == 0 or y.size == 0:
        return np.nan
    x = x.astype(float)
    y = y.astype(float)
    x_norm_value = norm(np.nan_to_num(x, nan=0.0))
    y_norm_value = norm(np.nan_to_num(y, nan=0.0))
    if x_norm_value == 0.0 or y_norm_value == 0.0:
        return np.nan
    return float(
        np.dot(np.nan_to_num(x, nan=0.0), np.nan_to_num(y, nan=0.0)) / (x_norm_value * y_norm_value))

# Compute subject/session state profiles & fidelity metrics:
profile_rows = []
fidelity_rows = []

for (subject_id, session_id), group_df in grouped_sequences:

    row_indices = group_df["_row_index"].to_numpy(dtype=int)
    windows_in_sequence = int(row_indices.size)

    responsibilities_seq = posterior_probability_matrix[row_indices, :]  # <-- (T_seq, K)
    raw_features_seq = feature_matrix_raw[row_indices, :]                # <-- (T_seq, F)

    responsibility_sums_seq = responsibilities_seq.sum(axis=0)          # <-- (K,)
    occupancy_soft_seq = responsibility_sums_seq / (np.nansum(responsibility_sums_seq) + epsilon)

    assigned_state_indices = np.argmax(responsibilities_seq, axis=1).astype(int)
    posterior_assigned_seq = responsibilities_seq[np.arange(windows_in_sequence), assigned_state_indices]

    for state_index in range(number_of_states):

        state_label = state_index_to_label[state_index]
        weights = responsibilities_seq[:, state_index].astype(float)
        weight_sum = float(np.sum(weights))

        if not np.isfinite(weight_sum) or weight_sum <= 0.0:
            fidelity_rows.append({
                "subject_ID": subject_id,
                "session_ID": session_id,
                "state_label": state_label,
                "windows_in_sequence": windows_in_sequence,
                "responsibility_sum": weight_sum,
                "occupancy_soft_within_sequence": float(occupancy_soft_seq[state_index]),
                "state_profile_correlation_with_global": np.nan,
                "state_profile_cosine_similarity_with_global": np.nan,
                "mean_posterior_probability_assigned_state": float(np.nanmean(posterior_assigned_seq)) if windows_in_sequence > 0 else np.nan})
            continue

        subject_state_mean_vector = (weights @ raw_features_seq) / (weight_sum + epsilon)  # (F,)
        global_state_mean_vector = state_feature_mean_df.loc[state_label, feature_columns_expected].to_numpy(dtype=float)

        state_profile_correlation = safe_pearson_correlation(subject_state_mean_vector, global_state_mean_vector)
        state_profile_cosine_similarity = safe_cosine_similarity(subject_state_mean_vector, global_state_mean_vector)

        fidelity_rows.append({
            "subject_ID": subject_id,
            "session_ID": session_id,
            "state_label": state_label,
            "windows_in_sequence": windows_in_sequence,
            "responsibility_sum": weight_sum,
            "occupancy_soft_within_sequence": float(occupancy_soft_seq[state_index]),
            "state_profile_correlation_with_global": state_profile_correlation,
            "state_profile_cosine_similarity_with_global": state_profile_cosine_similarity,
            "mean_posterior_probability_assigned_state": float(np.nanmean(posterior_assigned_seq)) if windows_in_sequence > 0 else np.nan})

        differences = subject_state_mean_vector - global_state_mean_vector
        zscores = differences / global_feature_std_safe_series.to_numpy(dtype=float)

        for feature_index, feature_name in enumerate(feature_columns_expected):
            z_value = zscores[feature_index]
            profile_rows.append({
                "subject_ID": subject_id,
                "session_ID": session_id,
                "state_label": state_label,
                "feature_name": feature_name,
                "subject_state_mean": float(subject_state_mean_vector[feature_index]),
                "global_state_mean": float(global_state_mean_vector[feature_index]),
                "difference_subject_minus_global_state_mean": float(differences[feature_index]),
                "zscore_subject_minus_global_state_mean": float(z_value) if np.isfinite(z_value) else np.nan})

# Assemble DataFrames:
subject_session_state_profile_long_df = pd.DataFrame(profile_rows)
subject_session_state_fidelity_df = pd.DataFrame(fidelity_rows)

subject_session_state_profile_long_df = subject_session_state_profile_long_df.sort_values(
    ["subject_ID", "session_ID", "state_label", "feature_name"],
    ascending=[True, True, True, True]).reset_index(drop=True)

subject_session_state_fidelity_df = subject_session_state_fidelity_df.sort_values(
    ["subject_ID", "session_ID", "state_label"], ascending=[True, True, True]).reset_index(drop=True)

# ------------------------------
# Export (guarded by OVERWRITE):
# ------------------------------
subject_session_state_profiles_long_path = OUTPUT_DIR / "subject_session_state_profiles_long.csv"
subject_session_state_fidelity_path = OUTPUT_DIR / "subject_session_state_fidelity.csv"

if subject_session_state_profiles_long_path.exists() and not OVERWRITE:
    print(f"[EXPORT] File exists; not overwriting (OVERWRITE=False): {subject_session_state_profiles_long_path}")
else:
    subject_session_state_profile_long_df.to_csv(subject_session_state_profiles_long_path, index=False)
    print(f"[EXPORT] Wrote: {subject_session_state_profiles_long_path} | shape={subject_session_state_profile_long_df.shape}")

if subject_session_state_fidelity_path.exists() and not OVERWRITE:
    print(f"[EXPORT] File exists; not overwriting (OVERWRITE=False): {subject_session_state_fidelity_path}")
else:
    subject_session_state_fidelity_df.to_csv(subject_session_state_fidelity_path, index=False)
    print(f"[EXPORT] Wrote: {subject_session_state_fidelity_path} | shape={subject_session_state_fidelity_df.shape}")

# ----------------
# Print reports:
# ----------------
print("\n[SUBJECT-LEVEL] Subject/session state profile tables created:")
print(f"  subject_session_state_profile_long_df   shape = {subject_session_state_profile_long_df.shape}")
print(f"  subject_session_state_fidelity_df       shape = {subject_session_state_fidelity_df.shape}")

print("\n[SUBJECT-LEVEL] Preview (fidelity metrics; first 12 rows):")
subject_session_state_fidelity_df.head(12)

-----------

We've now built & exported the 4 main "interim" data tables (see documentation at top of script); next we are collecting just the features we want to export for downstream ML, which is a narrower subset of all the various metrics/features we've computed so far:

In [ ]:
# __________________________________________________________________________________________________________
### SCRIPT #13 — ML-READY FEATURE HARVEST (SESSION-LEVEL OR SUBJECT-LEVEL VIA EXPORT_LEVEL)
#
# Inputs (must exist in memory from prior cells):
#   - subject_session_state_fidelity_df
#   - subject_session_state_profile_long_df
#   - OUTPUT_DIR (Path)
#   - OVERWRITE (bool)
#   - EXPORT_LEVEL (str): "session" or "subject"
#   - number_of_states (int)
#
# Output:
#   - hmm_script13_features_<export_level>.csv written to OUTPUT_DIR
#
# Notes:
#   - Uses verbose column naming conventions
#   - Avoids the "confidence" feature because it duplicates Script #12 optional mean_posterior_probability_max
#   - Implements subject-level collapsing only at the final feature-table stage (no need to redo earlier cells)
# __________________________________________________________________________________________________________

# Basic validation:
required_fidelity_columns = {
    "subject_ID",
    "session_ID",
    "state_label",
    "occupancy_soft_within_sequence",
    "state_profile_correlation_with_global",
    "state_profile_cosine_similarity_with_global"}
missing_fidelity_columns = sorted(list(required_fidelity_columns - set(subject_session_state_fidelity_df.columns)))
if missing_fidelity_columns:
    raise RuntimeError(
        "[FEATURE HARVEST ERROR] subject_session_state_fidelity_df is missing required columns:\n"
        + "\n".join([f"  - {column_name}" for column_name in missing_fidelity_columns]))

required_profile_long_columns = {
    "subject_ID",
    "session_ID",
    "state_label",
    "zscore_subject_minus_global_state_mean"}
missing_profile_long_columns = sorted(list(required_profile_long_columns - set(subject_session_state_profile_long_df.columns)))
if missing_profile_long_columns:
    raise RuntimeError(
        "[FEATURE HARVEST ERROR] subject_session_state_profile_long_df is missing required columns:\n"
        + "\n".join([f"  - {column_name}" for column_name in missing_profile_long_columns]))

if EXPORT_LEVEL not in {"session", "subject"}:
    raise ValueError(f"[FEATURE HARVEST ERROR] EXPORT_LEVEL must be 'session' or 'subject' (got: {EXPORT_LEVEL})")

# Helper for consistent state label formatting:
def format_state_string(state_value) -> str:
    """
    Converts state_label values into a consistent 'state-XX' format.

    Accepts:
      - strings: 'state_00', 'state_1', '0', etc.
      - ints: 0, 1, ...
    """
    if isinstance(state_value, str):
        state_string = state_value.strip()
        if state_string.lower().startswith("state_"):
            # e.g., state_00
            state_index = int(state_string.split("_", 1)[1])
            return f"state-{state_index:02d}"
        if state_string.isdigit():
            return f"state-{int(state_string):02d}"
        # fallback: try to parse trailing digits
        digits = "".join([character for character in state_string if character.isdigit()])
        if digits:
            return f"state-{int(digits):02d}"
        raise ValueError(f"Unrecognized state_label string format: {state_value}")
    else:
        return f"state-{int(state_value):02d}"


# Build per-[subject x session x state] core fidelity feature table:
fidelity_df = subject_session_state_fidelity_df.copy()

# Normalize 'state_label' formatting:
fidelity_df["state_label_formatted"] = fidelity_df["state_label"].apply(format_state_string)

# Re-name columns according to "verbose" naming conventions:
fidelity_df = fidelity_df.rename(columns={
    "occupancy_soft_within_sequence": "occupancy-Soft",
    "state_profile_correlation_with_global": "stateProfileCorr-Global",
    "state_profile_cosine_similarity_with_global": "stateProfileCosineSimilarity-Global"})

# Keep only what we are harvesting here (explicitly excluding confidence features):
fidelity_keep_columns = [
    "subject_ID",
    "session_ID",
    "state_label_formatted",
    "occupancy-Soft",
    "stateProfileCorr-Global",
    "stateProfileCosineSimilarity-Global"]
fidelity_df = fidelity_df[fidelity_keep_columns]

# Pivot wide; one row per [subject_ID x session_ID], w/ per-state columns:
fidelity_feature_columns = ["occupancy-Soft", "stateProfileCorr-Global", "stateProfileCosineSimilarity-Global"]

fidelity_wide_df = (
    fidelity_df
    .set_index(["subject_ID", "session_ID", "state_label_formatted"])[fidelity_feature_columns]
    .unstack("state_label_formatted"))

# Flatten multi-index columns (i.e. "<metric>_state-XX"):
fidelity_wide_df.columns = [
    f"{metric_name}_state-{state_string.split('-')[1]}"
    for metric_name, state_string in fidelity_wide_df.columns]
fidelity_wide_df = fidelity_wide_df.reset_index()

# Build 'meanAbsZdiff' per [subject x session x state] from long profile table:
profile_long_df = subject_session_state_profile_long_df.copy()
profile_long_df["state_label_formatted"] = profile_long_df["state_label"].apply(format_state_string)

# Compute mean absolute deviation across features, per [subject x session x state]:
profile_absz_agg_df = (
    profile_long_df
    .groupby(["subject_ID", "session_ID", "state_label_formatted"], dropna=False)["zscore_subject_minus_global_state_mean"]
    .apply(lambda series: float(np.mean(np.abs(series.to_numpy(dtype=float)))))
    .reset_index()
    .rename(columns={"zscore_subject_minus_global_state_mean": "meanAbsZdiff-Global"}))

# Pivot wide; one row per [subject_ID x session_ID]:
absz_wide_df = (
    profile_absz_agg_df
    .set_index(["subject_ID", "session_ID", "state_label_formatted"])[["meanAbsZdiff-Global"]]
    .unstack("state_label_formatted"))
absz_wide_df.columns = [
    f"meanAbsZdiff-Global_state-{state_string.split('-')[1]}"
    for (_, state_string) in absz_wide_df.columns]
absz_wide_df = absz_wide_df.reset_index()

# Merge into a single session-level feature table:
hmm_state_features_session_df = (
    fidelity_wide_df
    .merge(absz_wide_df, on=["subject_ID", "session_ID"], how="outer", validate="one_to_one"))

# Deterministic sorting:
hmm_state_features_session_df = hmm_state_features_session_df.sort_values(
    ["subject_ID", "session_ID"]).reset_index(drop=True)

print("\n[FEATURE HARVEST] Session-level Script #13 feature table created:")
print(f"  shape = {hmm_state_features_session_df.shape}")
# print("  preview (first 5 rows):")
# hmm_state_features_session_df.head()

# Optionally collapse to subject-level if EXPORT_LEVEL == "subject":
if EXPORT_LEVEL == "subject":
    # Use per-session window counts as weights if available; otherwise default to equal weights.
    # We infer n_windows from the long profile table by counting unique windows in the raw decoding,
    # but since we intentionally avoided cached Script #12 outputs here, we adopt a conservative strategy:
    # - If a 'number_of_windows' column exists in fidelity df, use that;
    # - otherwise, equal weights across sessions per subject.

    weight_column_name = None
    if "number_of_windows" in subject_session_state_fidelity_df.columns:
        weight_column_name = "number_of_windows"

    session_weight_df = (
        subject_session_state_fidelity_df[["subject_ID", "session_ID"] + ([weight_column_name] if weight_column_name else [])]
        .drop_duplicates().copy())

    if weight_column_name is None:
        session_weight_df["session_weight"] = 1.0
    else:
        session_weight_df["session_weight"] = pd.to_numeric(session_weight_df[weight_column_name], errors="coerce").fillna(1.0)

    # Attach weights to the session-level feature table
    weighted_df = hmm_state_features_session_df.merge(
        session_weight_df[["subject_ID", "session_ID", "session_weight"]],
        on=["subject_ID", "session_ID"],
        how="left",
        validate="one_to_one")

    feature_columns = [column_name for column_name in weighted_df.columns if column_name not in {"subject_ID", "session_ID", "session_weight"}]

    # Weighted mean aggregation across sessions per subject:
    #     --> Guardrails == if weights sum to 0 for a given subject, output NaNs for features
    def weighted_mean(series: pd.Series, weights: pd.Series) -> float:
        values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
        weight_values = pd.to_numeric(weights, errors="coerce").to_numpy(dtype=float)
        weight_sum = float(np.sum(weight_values))
        if not np.isfinite(weight_sum) or weight_sum <= 0:
            return float("nan")
        return float(np.nansum(values * weight_values) / weight_sum)

    aggregated_rows = []
    for subject_id, subject_df in weighted_df.groupby("subject_ID", dropna=False):
        weights = subject_df["session_weight"]
        aggregated_row = {"subject_ID": subject_id}
        for column_name in feature_columns:
            aggregated_row[column_name] = weighted_mean(subject_df[column_name], weights)
        aggregated_rows.append(aggregated_row)

    hmm_state_features_export_df = pd.DataFrame(aggregated_rows).sort_values(["subject_ID"]).reset_index(drop=True)

    # Note: 'session_ID' is now no longer meaningful at the subject level; hence we omit it to avoid ambiguity:
    print("\n[FEATURE HARVEST] EXPORT_LEVEL='subject' → collapsed session-level features to subject-level:")
    print(f"  shape = {hmm_state_features_export_df.shape}")
else:
    hmm_state_features_export_df = hmm_state_features_session_df.copy()

# ----------------------------------------
# Export (guarded by OVERWRITE setting):
# ----------------------------------------
output_filename = f"_flattened_HMM_state_features_ALL.csv"
output_path = OUTPUT_DIR / output_filename

if output_path.exists() and not OVERWRITE:
    print(f"\n[EXPORT] File exists; not overwriting (OVERWRITE=False): {output_path}")
else:
    hmm_state_features_export_df.to_csv(output_path, index=False)
    print(f"\n[EXPORT] Wrote: {output_path} | shape={hmm_state_features_export_df.shape}")

# -----------------------
# Print final report:
# -----------------------
print("\n[FEATURE HARVEST] Columns:")
print(list(hmm_state_features_export_df.columns))

print("\n[FEATURE HARVEST] Preview (first 10 rows):")
hmm_state_features_export_df.head(10)

-----------